# Data Fetching & Engineering

# 1. Configurations

In [1]:
from __future__ import annotations

import os
import re
from io import StringIO
from pathlib import Path
from urllib.parse import parse_qsl, urlsplit

import numpy as np
import pandas as pd
import requests

from dotenv import load_dotenv
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
ENV_PATH = PROJECT_ROOT / ".env"

if not ENV_PATH.exists():
    raise FileNotFoundError(f"Missing environment file: {ENV_PATH}")

# Ensures values in .env replace stale values left in the notebook kernel.
load_dotenv(ENV_PATH, override=True)

DATA_START = pd.Timestamp("2004-01-01")
RUN_AS_OF = (
    pd.Timestamp.now(tz="America/Toronto")
    .tz_localize(None)
    .normalize()
)

INFLATION_NOWCAST_DIR = (
    Path.home()
    / "Desktop"
    / "Data"
    / "Macro"
    / "Inflation_Nowcast"
)

INDPRO_PATH = Path.home() / "Desktop" / "INDPRO.csv"


def require_secret(name: str) -> str:
    value = os.getenv(name)

    if not value:
        raise RuntimeError(
            f"{name} is missing. Add it to {ENV_PATH}."
        )

    return value


# Validate configuration without printing the keys.
for secret_name in [
    "MASSIVE_API_KEY",
    "FRED_API_KEY",
    "TIINGO_API_KEY",
]:
    require_secret(secret_name)


def build_session(headers: dict | None = None) -> requests.Session:
    retry = Retry(
        total=4,
        connect=4,
        read=4,
        status=4,
        backoff_factor=0.75,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET"}),
        respect_retry_after_header=True,
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry,
        pool_connections=10,
        pool_maxsize=10,
    )

    session = requests.Session()
    session.mount("https://", adapter)

    if headers:
        session.headers.update(headers)

    return session


def request_json(
    session: requests.Session,
    url: str,
    *,
    params: dict | None = None,
    source: str,
    timeout: int = 60,
):
    try:
        response = session.get(
            url,
            params=params,
            timeout=timeout,
        )
    except requests.RequestException:
        raise RuntimeError(
            f"{source}: request failed; check connectivity."
        ) from None

    if not response.ok:
        raise RuntimeError(
            f"{source}: HTTP {response.status_code}."
        )

    try:
        return response.json()
    except ValueError:
        raise RuntimeError(
            f"{source}: response was not valid JSON."
        ) from None


def request_csv(
    session: requests.Session,
    url: str,
    *,
    source: str,
    timeout: int = 60,
) -> pd.DataFrame:
    try:
        response = session.get(url, timeout=timeout)
    except requests.RequestException:
        raise RuntimeError(
            f"{source}: request failed; check connectivity."
        ) from None

    if not response.ok:
        raise RuntimeError(
            f"{source}: HTTP {response.status_code}."
        )

    return pd.read_csv(StringIO(response.text))


print(
    f"Ingestion window: {DATA_START.date()} "
    f"through {RUN_AS_OF.date()}"
)

Ingestion window: 2004-01-01 through 2026-09-10


## Massive Data

In [3]:
MASSIVE_ENDPOINTS = {
    "treasury": "treasury-yields",
    "inflation": "inflation",
    "expectations": "inflation-expectations",
    "labor": "labor-market",
    "funding": "funding-conditions",
}


def load_massive_economy(
    endpoint: str,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
) -> pd.DataFrame:
    if endpoint not in MASSIVE_ENDPOINTS.values():
        raise ValueError(f"Unsupported Massive endpoint: {endpoint}")

    api_key = require_secret("MASSIVE_API_KEY")
    url = f"https://api.massive.com/fed/v1/{endpoint}"

    initial_params = {
        "apiKey": api_key,
        "date.gte": start_date.strftime("%Y-%m-%d"),
        "date.lte": end_date.strftime("%Y-%m-%d"),
        "limit": 50_000,
        "sort": "date.asc",
    }

    records = []
    first_request = True

    with build_session() as session:
        while url:
            if first_request:
                params = initial_params
                first_request = False
            else:
                existing_parameters = {
                    key for key, _ in parse_qsl(
                        urlsplit(url).query
                    )
                }

                params = (
                    None
                    if "apiKey" in existing_parameters
                    else {"apiKey": api_key}
                )

            payload = request_json(
                session,
                url,
                params=params,
                source=f"Massive {endpoint}",
            )

            batch = payload.get("results")

            if not isinstance(batch, list):
                raise RuntimeError(
                    f"Massive {endpoint}: missing results list."
                )

            records.extend(batch)
            next_url = payload.get("next_url")

            if next_url:
                next_host = urlsplit(next_url).hostname

                if next_host != "api.massive.com":
                    raise RuntimeError(
                        "Massive returned an unexpected pagination host."
                    )

            url = next_url

    if not records:
        raise RuntimeError(
            f"Massive {endpoint}: no observations returned."
        )

    frame = pd.DataFrame(records)

    if "date" not in frame.columns:
        raise RuntimeError(
            f"Massive {endpoint}: missing date field."
        )

    frame["date"] = (
        pd.to_datetime(frame["date"], utc=True, errors="raise")
        .dt.tz_localize(None)
        .dt.normalize()
    )

    if frame["date"].duplicated().any():
        duplicates = frame.loc[
            frame["date"].duplicated(False), "date"
        ].unique()

        raise ValueError(
            f"Massive {endpoint}: duplicate dates: "
            f"{duplicates[:5]}"
        )

    # Convert columns only when every nonmissing value is numeric.
    for column in frame.columns.difference(["date"]):
        original_nonmissing = frame[column].notna()
        converted = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

        if converted.loc[original_nonmissing].notna().all():
            frame[column] = converted

    return frame.set_index("date").sort_index()


massive_raw = {
    name: load_massive_economy(
        endpoint,
        DATA_START,
        RUN_AS_OF,
    )
    for name, endpoint in MASSIVE_ENDPOINTS.items()
}

# Convenient aliases.
massive_treasury_raw = massive_raw["treasury"]
massive_inflation_raw = massive_raw["inflation"]
massive_expectations_raw = massive_raw["expectations"]
massive_labor_raw = massive_raw["labor"]
massive_funding_raw = massive_raw["funding"]

massive_coverage = pd.DataFrame.from_dict(
    {
        name: {
            "rows": len(frame),
            "start": frame.index.min(),
            "end": frame.index.max(),
            "columns": ", ".join(frame.columns),
        }
        for name, frame in massive_raw.items()
    },
    orient="index",
)

display(massive_coverage)

,rows,start,end,columns
treasury,5675,2004-01-02,2026-09-08,"yield_1_month, yield_3_month, yield_1_year, yi..."
inflation,271,2004-01-01,2026-07-01,"cpi, cpi_year_over_year, cpi_core, pce, pce_co..."
expectations,272,2004-01-01,2026-08-01,"market_5_year, market_10_year, forward_years_5..."
labor,272,2004-01-01,2026-08-01,"unemployment_rate, labor_force_participation_r..."
funding,8289,2004-01-01,2026-09-10,"effective_fed_funds_rate, fed_overnight_repo_t..."


## FRED/ALFRED

In [11]:
FRED_SERIES = {
    # Growth and labor
    "ICSA": "ICSA",
    "unrate": "UNRATE",
    "sahm_realtime": "SAHMREALTIME",

    # Inflation and rates
    "cpi_index_sa": "CPIAUCSL",
    "T10YIE": "T10YIE",
    "DFII10": "DFII10",
    "infl_5y5y": "T5YIFR",

    # Policy and cash
    "fedfunds": "DFF",
    "cash_yield_3m": "TB3MS",

    # Financial conditions
    "NFCI": "NFCI",
    "ANFCI": "ANFCI",
    "HY_OAS": "BAMLH0A0HYM2",
    "EPU": "USEPUINDXD",

    # Commodities and currency
    "oil_wti": "DCOILWTICO",
    "usd_broad": "DTWEXBGS",
}


def fred_rows(
    endpoint: str,
    result_key: str,
    *,
    limit: int,
    **parameters,
) -> pd.DataFrame:
    params = {
        "api_key": require_secret("FRED_API_KEY"),
        "file_type": "json",
        "limit": limit,
        "offset": 0,
        **parameters,
    }

    records = []

    with build_session() as session:
        while True:
            payload = request_json(
                session,
                f"https://api.stlouisfed.org/fred/{endpoint}",
                params=params,
                source=f"FRED {endpoint}",
            )

            if result_key not in payload or "count" not in payload:
                raise RuntimeError(
                    f"FRED {endpoint}: unexpected response schema."
                )

            batch = payload[result_key]

            if not isinstance(batch, list):
                raise RuntimeError(
                    f"FRED {endpoint}: {result_key} is not a list."
                )

            records.extend(batch)
            params["offset"] += len(batch)

            if params["offset"] >= int(payload["count"]):
                break

            if not batch:
                raise RuntimeError(
                    f"FRED {endpoint}: incomplete pagination."
                )

    if not records:
        raise RuntimeError(
            f"FRED {endpoint}: no records returned."
        )

    return pd.DataFrame(records)


def fred_realtime_windows(
    series_id: str,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    max_vintages: int = 1_999,
) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    """
    Partition a FRED real-time period below the API's 2,000-vintage limit.

    FRED may count the starting snapshot as an additional vintage,
    so each window contains at most 1,999 listed vintage dates.
    """
    vintage_frame = fred_rows(
        "series/vintagedates",
        "vintage_dates",
        limit=10_000,
        series_id=series_id,
        realtime_start=start_date.strftime("%Y-%m-%d"),
        realtime_end=end_date.strftime("%Y-%m-%d"),
        sort_order="asc",
    )

    vintage_dates = (
        pd.DatetimeIndex(
            pd.to_datetime(
                vintage_frame.iloc[:, 0],
                errors="raise",
            )
        )
        .sort_values()
        .unique()
    )

    if len(vintage_dates) <= max_vintages:
        return [(start_date, end_date)]

    boundaries = list(
        vintage_dates[max_vintages::max_vintages]
    )

    window_starts = [start_date, *boundaries]
    window_ends = [
        boundary - pd.Timedelta(days=1)
        for boundary in boundaries
    ] + [end_date]

    windows = list(zip(window_starts, window_ends))

    assert all(start <= end for start, end in windows)

    return windows


def coalesce_fred_intervals(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    """
    Recombine identical validity intervals split only because the API
    requests were partitioned.
    """
    frame = frame.copy()

    frame["_realtime_start"] = pd.to_datetime(
        frame["realtime_start"],
        errors="raise",
    )
    frame["_realtime_end"] = pd.to_datetime(
        frame["realtime_end"],
        errors="raise",
    )

    frame = frame.sort_values(
        [
            "observation_date",
            "_realtime_start",
            "_realtime_end",
        ]
    ).reset_index(drop=True)

    previous_observation = frame["observation_date"].shift()
    previous_value = frame["value"].shift()
    previous_end = frame["_realtime_end"].shift()

    same_observation = frame["observation_date"].eq(
        previous_observation
    )

    same_value = (
        frame["value"].eq(previous_value)
        | (
            frame["value"].isna()
            & previous_value.isna()
        )
    )

    contiguous = frame["_realtime_start"].le(
        previous_end + pd.Timedelta(days=1)
    )

    frame["_interval_block"] = (
        ~(same_observation & same_value & contiguous)
    ).cumsum()

    frame = (
        frame.groupby(
            "_interval_block",
            sort=False,
            as_index=False,
        )
        .agg(
            observation_date=("observation_date", "first"),
            value=("value", "first"),
            _realtime_start=("_realtime_start", "min"),
            _realtime_end=("_realtime_end", "max"),
        )
    )

    frame["realtime_start"] = (
        frame["_realtime_start"].dt.strftime("%Y-%m-%d")
    )
    frame["realtime_end"] = (
        frame["_realtime_end"].dt.strftime("%Y-%m-%d")
    )

    return frame[
        [
            "observation_date",
            "realtime_start",
            "realtime_end",
            "value",
        ]
    ]


def load_fred_vintage_history(
    model_name: str,
    series_id: str,
    start_date: pd.Timestamp,
    as_of: pd.Timestamp,
) -> pd.DataFrame:
    windows = fred_realtime_windows(
        series_id=series_id,
        start_date=start_date,
        end_date=as_of,
    )

    parts = []

    for realtime_start, realtime_end in windows:
        part = fred_rows(
            "series/observations",
            "observations",
            limit=100_000,
            series_id=series_id,
            observation_start=start_date.strftime("%Y-%m-%d"),
            observation_end=as_of.strftime("%Y-%m-%d"),
            realtime_start=realtime_start.strftime("%Y-%m-%d"),
            realtime_end=realtime_end.strftime("%Y-%m-%d"),
            output_type=1,
            units="lin",
            sort_order="asc",
        )

        parts.append(part)

    frame = pd.concat(parts, ignore_index=True)

    required = {
        "date",
        "value",
        "realtime_start",
        "realtime_end",
    }

    missing = required.difference(frame.columns)

    if missing:
        raise RuntimeError(
            f"FRED {series_id}: missing fields {sorted(missing)}."
        )

    frame = frame.rename(
        columns={"date": "observation_date"}
    )

    frame["observation_date"] = pd.to_datetime(
        frame["observation_date"],
        errors="raise",
    )

    frame["value"] = pd.to_numeric(
        frame["value"].replace(".", np.nan),
        errors="raise",
    )

    frame = coalesce_fred_intervals(frame)

    frame.insert(0, "series", model_name)
    frame.insert(1, "series_id", series_id)

    frame = frame[
        [
            "series",
            "series_id",
            "observation_date",
            "realtime_start",
            "realtime_end",
            "value",
        ]
    ].sort_values(
        [
            "observation_date",
            "realtime_start",
        ]
    )

    duplicate_mask = frame.duplicated(
        [
            "observation_date",
            "realtime_start",
            "realtime_end",
        ]
    )

    if duplicate_mask.any():
        raise ValueError(
            f"FRED {series_id}: duplicate vintage records."
        )

    print(
        f"{series_id}: "
        f"{len(frame):,} records across "
        f"{len(windows)} request window(s)"
    )

    return frame.reset_index(drop=True)


fred_vintage_history = pd.concat(
    [
        load_fred_vintage_history(
            model_name=model_name,
            series_id=series_id,
            start_date=DATA_START,
            as_of=RUN_AS_OF,
        )
        for model_name, series_id in FRED_SERIES.items()
    ],
    ignore_index=True,
).sort_values(
    [
        "series",
        "observation_date",
        "realtime_start",
    ]
).reset_index(drop=True)


def fred_snapshot(
    history: pd.DataFrame,
    as_of: pd.Timestamp,
) -> pd.DataFrame:
    """Return every FRED series exactly as recorded at a date."""
    cutoff = pd.Timestamp(as_of).normalize()
    day = cutoff.strftime("%Y-%m-%d")

    active = history.loc[
        history["realtime_start"].le(day)
        & history["realtime_end"].ge(day)
        & history["observation_date"].le(cutoff)
    ].copy()

    if active.duplicated(
        ["series", "observation_date"]
    ).any():
        raise ValueError(
            f"Overlapping FRED vintages at {day}."
        )

    return (
        active.pivot(
            index="observation_date",
            columns="series",
            values="value",
        )
        .sort_index()
        .rename_axis(index="date", columns=None)
    )


fred_asof = fred_snapshot(
    fred_vintage_history,
    RUN_AS_OF,
)

# Compatibility alias for later work.
fred_raw = fred_asof


# CPI release calendar used to audit first-vintage dates.
cpi_release_dates = fred_rows(
    "release/dates",
    "release_dates",
    limit=10_000,
    release_id=10,
    realtime_start=DATA_START.strftime("%Y-%m-%d"),
    realtime_end=RUN_AS_OF.strftime("%Y-%m-%d"),
    include_release_dates_with_no_data="false",
    sort_order="asc",
)

cpi_release_dates["date"] = pd.to_datetime(
    cpi_release_dates["date"],
    errors="raise",
)

if cpi_release_dates["date"].duplicated().any():
    raise ValueError("Duplicate CPI release dates.")

fred_coverage = (
    fred_vintage_history
    .groupby(["series", "series_id"])
    .agg(
        vintage_records=("value", "size"),
        nonmissing_records=("value", "count"),
        first_observation=("observation_date", "min"),
        last_observation=("observation_date", "max"),
        first_available=("realtime_start", "min"),
    )
)

display(fred_coverage)

ICSA: 5,861 records across 1 request window(s)
UNRATE: 506 records across 1 request window(s)
SAHMREALTIME: 272 records across 1 request window(s)
CPIAUCSL: 1,412 records across 1 request window(s)
T10YIE: 5,923 records across 2 request window(s)
DFII10: 5,921 records across 3 request window(s)
T5YIFR: 5,924 records across 2 request window(s)
DFF: 8,314 records across 3 request window(s)
TB3MS: 272 records across 1 request window(s)
NFCI: 338,458 records across 1 request window(s)
ANFCI: 377,710 records across 1 request window(s)
BAMLH0A0HYM2: 795 records across 1 request window(s)
USEPUINDXD: 117,957 records across 2 request window(s)
DCOILWTICO: 6,013 records across 1 request window(s)
DTWEXBGS: 22,072 records across 1 request window(s)


,,vintage_records,nonmissing_records,first_observation,last_observation,first_available
series,series_id,,,,,
ANFCI,ANFCI,377710,377710,2004-01-02,2026-09-04,2011-05-25
DFII10,DFII10,5921,5676,2004-01-01,2026-09-09,2005-10-12
EPU,USEPUINDXD,117957,117878,2004-01-01,2026-09-09,2014-03-27
HY_OAS,BAMLH0A0HYM2,795,787,2023-09-11,2026-09-09,2023-09-11
ICSA,ICSA,5861,5859,2004-01-03,2026-09-05,2009-05-28
NFCI,NFCI,338458,338458,2004-01-02,2026-09-04,2011-05-25
T10YIE,T10YIE,5923,5679,2004-01-01,2026-09-10,2014-01-27
cash_yield_3m,TB3MS,272,272,2004-01-01,2026-08-01,2004-02-02
cpi_index_sa,CPIAUCSL,1412,1411,2004-01-01,2026-07-01,2004-02-20


## CBOE VIX

In [5]:
CBOE_VIX_URL = (
    "https://cdn.cboe.com/api/global/us_indices/"
    "daily_prices/VIX_History.csv"
)


with build_session() as session:
    vix_daily = request_csv(
        session,
        CBOE_VIX_URL,
        source="Cboe VIX",
    )

vix_daily.columns = (
    vix_daily.columns
    .str.strip()
    .str.lower()
)

required_vix_columns = {
    "date",
    "open",
    "high",
    "low",
    "close",
}

missing_vix_columns = required_vix_columns.difference(
    vix_daily.columns
)

if missing_vix_columns:
    raise ValueError(
        f"Cboe VIX missing fields: {sorted(missing_vix_columns)}"
    )

vix_daily["date"] = pd.to_datetime(
    vix_daily["date"],
    errors="raise",
)

for column in ["open", "high", "low", "close"]:
    vix_daily[column] = pd.to_numeric(
        vix_daily[column],
        errors="raise",
    )

vix_daily = (
    vix_daily.loc[
        vix_daily["date"].le(RUN_AS_OF),
        ["date", "open", "high", "low", "close"],
    ]
    .rename(
        columns={
            "open": "vix_open",
            "high": "vix_high",
            "low": "vix_low",
            "close": "vix_close",
        }
    )
    .sort_values("date")
)

if vix_daily["date"].duplicated().any():
    raise ValueError("Duplicate Cboe VIX dates.")

vix_daily = vix_daily.set_index("date")

if (
    vix_daily.empty
    or not np.isfinite(vix_daily).all().all()
    or vix_daily.le(0).any().any()
):
    raise ValueError("Invalid Cboe VIX data.")

# Compatibility series.
vix = vix_daily["vix_close"].rename("VIX")

display(vix_daily.tail())

,vix_open,vix_high,vix_low,vix_close
date,,,,
2026-09-03,15.25,15.44,14.23,14.32
2026-09-04,14.15,14.58,13.80,14.53
2026-09-07,15.02,15.32,14.99,15.30
2026-09-08,15.56,15.94,15.22,15.72
2026-09-09,15.65,16.68,15.57,16.46


## Cleveland Nowcasts

In [6]:
INFLATION_NOWCAST_COLUMNS = {
    "CPI Inflation": "cpi_inflation_mom",
    "Core CPI Inflation": "core_cpi_inflation_mom",
    "PCE Inflation": "pce_inflation_mom",
    "Core PCE Inflation": "core_pce_inflation_mom",
}

INFLATION_VALUE_COLUMNS = list(
    INFLATION_NOWCAST_COLUMNS.values()
)


def parse_target_month(file_path: Path) -> pd.Timestamp:
    match = re.fullmatch(
        (
            r"Month-Over-MonthPercentChange-"
            r"(\d{4})-(\d{1,2})\.csv"
        ),
        file_path.name,
    )

    if match is None:
        raise ValueError(
            f"Unexpected nowcast filename: {file_path.name}"
        )

    return pd.Timestamp(
        year=int(match.group(1)),
        month=int(match.group(2)),
        day=1,
    )


def parse_nowcast_dates(
    labels: pd.Series,
    target_month: pd.Timestamp,
) -> pd.DatetimeIndex:
    dates = []
    current_year = target_month.year
    previous_month = None

    for label in labels:
        parts = str(label).strip().split("/")

        if len(parts) != 2:
            raise ValueError(
                f"Invalid nowcast date label: {label}"
            )

        month, day = map(int, parts)

        if (
            previous_month is not None
            and month < previous_month
        ):
            current_year += 1

        dates.append(
            pd.Timestamp(
                year=current_year,
                month=month,
                day=day,
            )
        )

        previous_month = month

    dates = pd.DatetimeIndex(dates)

    if dates.has_duplicates:
        raise ValueError("Duplicate dates inside a nowcast file.")

    if not dates.is_monotonic_increasing:
        raise ValueError(
            "Nowcast dates are not chronologically ordered."
        )

    month_offsets = (
        dates.to_period("M").astype("int64")
        - target_month.to_period("M").ordinal
    )

    if (month_offsets < 0).any() or (month_offsets > 3).any():
        raise ValueError(
            "Forecast dates fall outside the expected target window."
        )

    return dates


def read_inflation_nowcast_file(
    file_path: Path,
) -> pd.DataFrame:
    target_month = parse_target_month(file_path)

    frame = pd.read_csv(file_path)
    frame.columns = frame.columns.str.strip()

    required = [
        "Label",
        *INFLATION_NOWCAST_COLUMNS.keys(),
    ]

    missing = set(required).difference(frame.columns)

    if missing:
        raise ValueError(
            f"{file_path.name} missing fields: {sorted(missing)}"
        )

    frame = frame[required].rename(
        columns=INFLATION_NOWCAST_COLUMNS
    )

    frame["forecast_date"] = parse_nowcast_dates(
        frame["Label"],
        target_month,
    )

    frame["target_month"] = target_month
    frame["source_file"] = file_path.name

    for column in INFLATION_VALUE_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

        if np.isinf(frame[column]).any():
            raise ValueError(
                f"{file_path.name}: infinite {column}."
            )

    return frame[
        [
            "forecast_date",
            "target_month",
            "source_file",
            *INFLATION_VALUE_COLUMNS,
        ]
    ]


def load_inflation_nowcasts(
    folder: Path,
    as_of: pd.Timestamp,
) -> pd.DataFrame:
    files = sorted(
        folder.glob(
            "Month-Over-MonthPercentChange-*.csv"
        ),
        key=parse_target_month,
    )

    if not files:
        raise FileNotFoundError(
            f"No inflation nowcast files in {folder}"
        )

    combined = pd.concat(
        [
            read_inflation_nowcast_file(file_path)
            for file_path in files
        ],
        ignore_index=True,
    )

    combined = combined.loc[
        combined["forecast_date"].le(as_of)
    ].sort_values(
        ["forecast_date", "target_month"]
    )

    if combined.duplicated(
        ["forecast_date", "target_month"]
    ).any():
        raise ValueError(
            "Duplicate forecast-date/target-month pairs."
        )

    return combined.set_index(
        ["forecast_date", "target_month"]
    ).sort_index()


inflation_nowcast_long = load_inflation_nowcasts(
    INFLATION_NOWCAST_DIR,
    RUN_AS_OF,
)

inflation_nowcast_coverage = pd.Series(
    {
        "files": (
            inflation_nowcast_long["source_file"].nunique()
        ),
        "rows": len(inflation_nowcast_long),
        "first_forecast_date": (
            inflation_nowcast_long.index
            .get_level_values("forecast_date")
            .min()
        ),
        "last_forecast_date": (
            inflation_nowcast_long.index
            .get_level_values("forecast_date")
            .max()
        ),
        "first_target_month": (
            inflation_nowcast_long.index
            .get_level_values("target_month")
            .min()
        ),
        "last_target_month": (
            inflation_nowcast_long.index
            .get_level_values("target_month")
            .max()
        ),
    },
    name="Cleveland inflation nowcasts",
)

display(inflation_nowcast_coverage)
display(inflation_nowcast_long.tail())

files                                  159
rows                                  6567
first_forecast_date    2013-08-20 00:00:00
last_forecast_date     2026-09-09 00:00:00
first_target_month     2013-07-01 00:00:00
last_target_month      2026-09-01 00:00:00
Name: Cleveland inflation nowcasts, dtype: object

source_file  \
forecast_date target_month                                             
2026-09-04    2026-09-01    Month-Over-MonthPercentChange-2026-9.csv   
2026-09-08    2026-08-01    Month-Over-MonthPercentChange-2026-8.csv   
              2026-09-01    Month-Over-MonthPercentChange-2026-9.csv   
2026-09-09    2026-08-01    Month-Over-MonthPercentChange-2026-8.csv   
              2026-09-01    Month-Over-MonthPercentChange-2026-9.csv   

                            cpi_inflation_mom  core_cpi_inflation_mom  \
forecast_date target_month                                              
2026-09-04    2026-09-01             0.382154                0.194477   
2026-09-08    2026-08-01             0.359181                0.203342   
              2026-09-01             0.396439                0.194477   
2026-09-09    2026-08-01             0.359181                0.203342   
              2026-09-01             0.405558                0.194477   

                            pce_inflation_mom  core_pce_inflation_mom  
forecast_date target_month                                             
2026-09-04    2026-09-01             0.375605                0.278693  
2026-09-08    2026-08-01             0.353657                0.274535  
              2026-09-01             0.383808                0.278693  
2026-09-09    2026-08-01             0.353657                0.274535  
              2026-09-01             0.389045                0.278693

## INDPRO

In [7]:
if not INDPRO_PATH.exists():
    raise FileNotFoundError(f"Missing file: {INDPRO_PATH}")

indpro_vintages = pd.read_csv(
    INDPRO_PATH,
    index_col="observation_date",
    parse_dates=["observation_date"],
    na_values=["."],
)

if not indpro_vintages.columns.str.fullmatch(
    r"INDPRO_\d{8}"
).all():
    raise ValueError("Unexpected INDPRO vintage columns.")

indpro_vintages.columns = pd.DatetimeIndex(
    pd.to_datetime(
        indpro_vintages.columns.str.replace(
            "INDPRO_",
            "",
            regex=False,
        ),
        format="%Y%m%d",
        errors="raise",
    ),
    name="vintage_date",
)

# Respect the ingestion cutoff if the local file contains later vintages.
indpro_vintages = indpro_vintages.loc[
    :,
    indpro_vintages.columns <= RUN_AS_OF,
]

if indpro_vintages.shape[1] == 0:
    raise ValueError(
        "No INDPRO vintage exists by the ingestion cutoff."
    )

if (
    indpro_vintages.index.has_duplicates
    or indpro_vintages.columns.has_duplicates
):
    raise ValueError(
        "Duplicate INDPRO observation or vintage dates."
    )

indpro_vintages = (
    indpro_vintages
    .apply(pd.to_numeric, errors="raise")
    .sort_index()
    .sort_index(axis=1)
    .asfreq("MS")
)

indpro_vintages.index.name = "observation_date"

indpro_long = (
    indpro_vintages
    .reset_index()
    .melt(
        id_vars="observation_date",
        var_name="vintage_date",
        value_name="INDPRO",
    )
    .dropna(subset=["INDPRO"])
    .sort_values(["vintage_date", "observation_date"])
    .reset_index(drop=True)
)

indpro_long["vintage_date"] = pd.to_datetime(
    indpro_long["vintage_date"]
)

if (
    not np.isfinite(indpro_long["INDPRO"]).all()
    or indpro_long["INDPRO"].le(0).any()
):
    raise ValueError("Invalid INDPRO index levels.")

indpro_latest_vintage = indpro_vintages.columns.max()

indpro_latest = (
    indpro_vintages[indpro_latest_vintage]
    .dropna()
    .rename("INDPRO")
    .to_frame()
)

print(
    f"{len(indpro_long):,} INDPRO vintage records | "
    f"{indpro_vintages.shape[1]} vintages | "
    f"latest {indpro_latest_vintage.date()}"
)

display(indpro_latest.tail())

12,092 INDPRO vintage records | 162 vintages | latest 2026-08-18


,INDPRO
observation_date,
2026-03-01,101.7543
2026-04-01,102.5198
2026-05-01,102.5099
2026-06-01,102.7868
2026-07-01,102.9939


## Tiingo ETF series

In [8]:
ETF_TICKERS = [
    "SPY",
    "IEF",
    "TLT",
    "LQD",
    "HYG",
    "TIP",
]


def load_tiingo_ticker(
    ticker: str,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    session: requests.Session,
) -> pd.DataFrame:
    payload = request_json(
        session,
        (
            "https://api.tiingo.com/tiingo/daily/"
            f"{ticker}/prices"
        ),
        params={
            "startDate": start_date.strftime("%Y-%m-%d"),
            "endDate": end_date.strftime("%Y-%m-%d"),
            "resampleFreq": "daily",
            "format": "json",
        },
        source=f"Tiingo {ticker}",
    )

    if not isinstance(payload, list) or not payload:
        raise RuntimeError(
            f"Tiingo {ticker}: no observations returned."
        )

    frame = pd.DataFrame(payload)

    required = {
        "date",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "adjOpen",
        "adjHigh",
        "adjLow",
        "adjClose",
        "adjVolume",
        "divCash",
        "splitFactor",
    }

    missing = required.difference(frame.columns)

    if missing:
        raise ValueError(
            f"Tiingo {ticker} missing fields: {sorted(missing)}"
        )

    frame = frame.rename(
        columns={
            "adjOpen": "adj_open",
            "adjHigh": "adj_high",
            "adjLow": "adj_low",
            "adjClose": "adj_close",
            "adjVolume": "adj_volume",
            "divCash": "div_cash",
            "splitFactor": "split_factor",
        }
    )

    frame["date"] = (
        pd.to_datetime(
            frame["date"],
            utc=True,
            errors="raise",
        )
        .dt.tz_localize(None)
        .dt.normalize()
    )

    numeric_columns = [
        "open",
        "high",
        "low",
        "close",
        "volume",
        "adj_open",
        "adj_high",
        "adj_low",
        "adj_close",
        "adj_volume",
        "div_cash",
        "split_factor",
    ]

    for column in numeric_columns:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="raise",
        )

    if frame["date"].duplicated().any():
        raise ValueError(f"Tiingo {ticker}: duplicate dates.")

    if (
        not np.isfinite(frame["adj_close"]).all()
        or frame["adj_close"].le(0).any()
    ):
        raise ValueError(
            f"Tiingo {ticker}: invalid adjusted prices."
        )

    frame.insert(1, "ticker", ticker)

    return frame[
        ["date", "ticker", *numeric_columns]
    ].sort_values("date")


tiingo_headers = {
    "Authorization": (
        f"Token {require_secret('TIINGO_API_KEY')}"
    ),
    "Content-Type": "application/json",
}

with build_session(tiingo_headers) as tiingo_session:
    tiingo_frames = [
        load_tiingo_ticker(
            ticker,
            DATA_START,
            RUN_AS_OF,
            tiingo_session,
        )
        for ticker in ETF_TICKERS
    ]

tiingo_etf_daily = (
    pd.concat(tiingo_frames, ignore_index=True)
    .set_index(["date", "ticker"])
    .sort_index()
)

if not tiingo_etf_daily.index.is_unique:
    raise ValueError(
        "Duplicate Tiingo date/ticker observations."
    )

# Adjusted price levels; returns are constructed in a later layer.
etf_adjusted_close = (
    tiingo_etf_daily["adj_close"]
    .unstack("ticker")
    .reindex(columns=ETF_TICKERS)
    .sort_index()
)

etf_raw_close = (
    tiingo_etf_daily["close"]
    .unstack("ticker")
    .reindex(columns=ETF_TICKERS)
    .sort_index()
)

etf_dividends = (
    tiingo_etf_daily["div_cash"]
    .unstack("ticker")
    .reindex(columns=ETF_TICKERS)
    .sort_index()
)

etf_split_factors = (
    tiingo_etf_daily["split_factor"]
    .unstack("ticker")
    .reindex(columns=ETF_TICKERS)
    .sort_index()
)

display(etf_adjusted_close.tail())

ticker,SPY,IEF,TLT,LQD,HYG,TIP
date,,,,,,
2026-09-03,773.17,92.280,82.07,105.50,79.21,107.00
2026-09-04,770.19,92.250,82.21,105.48,79.16,106.97
2026-09-08,765.96,92.160,82.20,105.48,79.12,107.05
2026-09-09,762.40,91.895,81.73,105.31,78.98,106.80
2026-09-10,757.83,91.180,80.78,104.36,78.62,106.33


## Final Data Coverage

In [12]:
raw_data = {
    "massive": massive_raw,
    "fred_vintage_history": fred_vintage_history,
    "fred_asof": fred_asof,
    "cpi_release_dates": cpi_release_dates,
    "cboe_vix_daily": vix_daily,
    "cleveland_inflation_nowcasts": inflation_nowcast_long,
    "indpro_vintages": indpro_vintages,
    "indpro_long": indpro_long,
    "tiingo_etf_daily": tiingo_etf_daily,
}


def coverage_record(
    dataset: str,
    source: str,
    rows: int,
    dates,
    detail: str = "",
) -> dict:
    dates = pd.DatetimeIndex(
        pd.to_datetime(dates)
    ).dropna()

    return {
        "dataset": dataset,
        "source": source,
        "rows": rows,
        "start": dates.min() if len(dates) else pd.NaT,
        "end": dates.max() if len(dates) else pd.NaT,
        "detail": detail,
    }


coverage_records = []

for name, frame in massive_raw.items():
    coverage_records.append(
        coverage_record(
            dataset=f"massive_{name}",
            source="Massive",
            rows=len(frame),
            dates=frame.index,
            detail=f"{frame.shape[1]} columns",
        )
    )

for series_name, group in fred_vintage_history.groupby(
    "series",
    sort=True,
):
    coverage_records.append(
        coverage_record(
            dataset=series_name,
            source="FRED/ALFRED",
            rows=len(group),
            dates=group["observation_date"],
            detail=(
                f"{group['value'].notna().sum()} "
                "nonmissing vintage records"
            ),
        )
    )

coverage_records.extend(
    [
        coverage_record(
            dataset="cpi_release_calendar",
            source="FRED",
            rows=len(cpi_release_dates),
            dates=cpi_release_dates["date"],
        ),
        coverage_record(
            dataset="vix",
            source="Cboe",
            rows=len(vix_daily),
            dates=vix_daily.index,
        ),
        coverage_record(
            dataset="inflation_nowcasts",
            source="Cleveland Fed",
            rows=len(inflation_nowcast_long),
            dates=(
                inflation_nowcast_long.index
                .get_level_values("forecast_date")
            ),
            detail=(
                f"{inflation_nowcast_long.index.get_level_values('target_month').nunique()} "
                "target months"
            ),
        ),
        coverage_record(
            dataset="INDPRO",
            source="Downloaded ALFRED file",
            rows=len(indpro_long),
            dates=indpro_long["observation_date"],
            detail=(
                f"{indpro_vintages.shape[1]} vintages"
            ),
        ),
    ]
)

for ticker, group in (
    tiingo_etf_daily
    .reset_index()
    .groupby("ticker", sort=True)
):
    coverage_records.append(
        coverage_record(
            dataset=ticker,
            source="Tiingo",
            rows=len(group),
            dates=group["date"],
            detail="adjusted and unadjusted OHLCV",
        )
    )

data_coverage = (
    pd.DataFrame(coverage_records)
    .set_index(["source", "dataset"])
    .sort_index()
)

# Final structural assertions.
assert all(
    not frame.empty
    for frame in massive_raw.values()
)

assert not fred_vintage_history.empty
assert not fred_asof.empty
assert not vix_daily.empty
assert not inflation_nowcast_long.empty
assert not indpro_long.empty
assert not tiingo_etf_daily.empty

assert all(
    frame.index.is_unique
    for frame in massive_raw.values()
)

assert vix_daily.index.is_unique
assert inflation_nowcast_long.index.is_unique
assert indpro_vintages.index.is_unique
assert indpro_vintages.columns.is_unique
assert tiingo_etf_daily.index.is_unique

print(f"Raw ingestion complete as of {RUN_AS_OF.date()}.")
display(data_coverage)

Raw ingestion complete as of 2026-09-10.


rows      start        end  \
source                 dataset                                              
Cboe                   vix                     9269 1990-01-02 2026-09-09   
Cleveland Fed          inflation_nowcasts      6567 2013-08-20 2026-09-09   
Downloaded ALFRED file INDPRO                 12092 2014-01-01 2026-07-01   
FRED                   cpi_release_calendar     289 2004-01-15 2026-08-12   
FRED/ALFRED            ANFCI                 377710 2004-01-02 2026-09-04   
                       DFII10                  5921 2004-01-01 2026-09-09   
                       EPU                   117957 2004-01-01 2026-09-09   
                       HY_OAS                   795 2023-09-11 2026-09-09   
                       ICSA                    5861 2004-01-03 2026-09-05   
                       NFCI                  338458 2004-01-02 2026-09-04   
                       T10YIE                  5923 2004-01-01 2026-09-10   
                       cash_yield_3m            272 2004-01-01 2026-08-01   
                       cpi_index_sa            1412 2004-01-01 2026-07-01   
                       fedfunds                8314 2004-01-01 2026-09-09   
                       infl_5y5y               5924 2004-01-01 2026-09-10   
                       oil_wti                 6013 2004-01-01 2026-09-09   
                       sahm_realtime            272 2004-01-01 2026-08-01   
                       unrate                   506 2004-01-01 2026-08-01   
                       usd_broad              22072 2006-01-02 2026-09-04   
Massive                massive_expectations     272 2004-01-01 2026-08-01   
                       massive_funding         8289 2004-01-01 2026-09-10   
                       massive_inflation        271 2004-01-01 2026-07-01   
                       massive_labor            272 2004-01-01 2026-08-01   
                       massive_treasury        5675 2004-01-02 2026-09-08   
Tiingo                 HYG                     4886 2007-04-11 2026-09-10   
                       IEF                     5708 2004-01-02 2026-09-10   
                       LQD                     5708 2004-01-02 2026-09-10   
                       SPY                     5708 2004-01-02 2026-09-10   
                       TIP                     5708 2004-01-02 2026-09-10   
                       TLT                     5708 2004-01-02 2026-09-10   

                                                                        detail  
source                 dataset                                                  
Cboe                   vix                                                      
Cleveland Fed          inflation_nowcasts                    159 target months  
Downloaded ALFRED file INDPRO                                     162 vintages  
FRED                   cpi_release_calendar                                     
FRED/ALFRED            ANFCI                 377710 nonmissing vintage records  
                       DFII10                  5676 nonmissing vintage records  
                       EPU                   117878 nonmissing vintage records  
                       HY_OAS                   787 nonmissing vintage records  
                       ICSA                    5859 nonmissing vintage records  
                       NFCI                  338458 nonmissing vintage records  
                       T10YIE                  5679 nonmissing vintage records  
                       cash_yield_3m            272 nonmissing vintage records  
                       cpi_index_sa            1411 nonmissing vintage records  
                       fedfunds                8314 nonmissing vintage records  
                       infl_5y5y               5680 nonmissing vintage records  
                       oil_wti                 5758 nonmissing vintage records  
                       sahm_realtime            271 nonmissing vintage records  
                       unrate              

# 2. Point in time Calendar

In [13]:
# 1. Data contracts and weekly decision calendar
# ============================================================

RUN_CUTOFF = pd.Timestamp(RUN_AS_OF).normalize()

DATASET_CONTRACTS = pd.DataFrame(
    [
        {
            "dataset": "massive_treasury",
            "source": "Massive",
            "record_type": "observation",
            "availability_rule": "Observation date, end of day",
            "pit_eligible": True,
            "model_role": "macro feature candidate",
        },
        {
            "dataset": "massive_funding",
            "source": "Massive",
            "record_type": "observation",
            "availability_rule": "Observation date, end of day",
            "pit_eligible": True,
            "model_role": "macro feature candidate",
        },
        {
            "dataset": "massive_inflation",
            "source": "Massive",
            "record_type": "current-history snapshot",
            "availability_rule": "Historical release date unavailable",
            "pit_eligible": False,
            "model_role": "reference and cross-check only",
        },
        {
            "dataset": "massive_expectations",
            "source": "Massive",
            "record_type": "current-history snapshot",
            "availability_rule": "Historical release date unavailable",
            "pit_eligible": False,
            "model_role": "reference and cross-check only",
        },
        {
            "dataset": "massive_labor",
            "source": "Massive",
            "record_type": "current-history snapshot",
            "availability_rule": "Historical release date unavailable",
            "pit_eligible": False,
            "model_role": "reference and cross-check only",
        },
        {
            "dataset": "fred_alfred",
            "source": "FRED/ALFRED",
            "record_type": "vintage interval",
            "availability_rule": "ALFRED realtime_start",
            "pit_eligible": True,
            "model_role": "macro feature candidate",
        },
        {
            "dataset": "indpro",
            "source": "Downloaded ALFRED file",
            "record_type": "vintage interval",
            "availability_rule": "Vintage-column date",
            "pit_eligible": True,
            "model_role": "growth feature candidate",
        },
        {
            "dataset": "cleveland_nowcast",
            "source": "Cleveland Fed",
            "record_type": "forecast update",
            "availability_rule": "Forecast publication date",
            "pit_eligible": True,
            "model_role": "inflation feature candidate",
        },
        {
            "dataset": "cboe_vix",
            "source": "Cboe",
            "record_type": "market close",
            "availability_rule": "Trading date, after close",
            "pit_eligible": True,
            "model_role": "stress feature candidate",
        },
        {
            "dataset": "tiingo_etf",
            "source": "Tiingo",
            "record_type": "market close",
            "availability_rule": "Trading date, after close",
            "pit_eligible": True,
            "model_role": "evaluation target",
        },
    ]
).set_index("dataset")


# A regime estimate is formed after Friday's market close.
# The corresponding investment return begins on the next SPY trading day.
latest_required_market_date = min(
    vix_daily.index.max(),
    etf_adjusted_close["SPY"].dropna().index.max(),
    massive_treasury_raw.index.max(),
    massive_funding_raw.index.max(),
)

decision_dates = pd.date_range(
    start=DATA_START,
    end=latest_required_market_date,
    freq="W-FRI",
)

spy_dates = pd.DatetimeIndex(
    etf_adjusted_close["SPY"].dropna().index
).sort_values()

next_trade_positions = spy_dates.searchsorted(
    decision_dates,
    side="right",
)

has_next_trade = next_trade_positions < len(spy_dates)

decision_dates = decision_dates[has_next_trade]
next_trade_positions = next_trade_positions[has_next_trade]

decision_calendar = pd.DataFrame(
    {
        "decision_date": decision_dates,
        "decision_at": (
            decision_dates + pd.Timedelta(hours=16)
        ).tz_localize("America/New_York"),
        "next_trade_date": spy_dates[next_trade_positions],
    }
)

decision_calendar["information_lag_days"] = (
    decision_calendar["next_trade_date"]
    - decision_calendar["decision_date"]
).dt.days

assert decision_calendar["decision_date"].dt.dayofweek.eq(4).all()
assert (
    decision_calendar["next_trade_date"]
    > decision_calendar["decision_date"]
).all()

display(DATASET_CONTRACTS)
display(decision_calendar.tail())

,source,record_type,availability_rule,pit_eligible,model_role
dataset,,,,,
massive_treasury,Massive,observation,"Observation date, end of day",True,macro feature candidate
massive_funding,Massive,observation,"Observation date, end of day",True,macro feature candidate
massive_inflation,Massive,current-history snapshot,Historical release date unavailable,False,reference and cross-check only
massive_expectations,Massive,current-history snapshot,Historical release date unavailable,False,reference and cross-check only
massive_labor,Massive,current-history snapshot,Historical release date unavailable,False,reference and cross-check only
fred_alfred,FRED/ALFRED,vintage interval,ALFRED realtime_start,True,macro feature candidate
indpro,Downloaded ALFRED file,vintage interval,Vintage-column date,True,growth feature candidate
cleveland_nowcast,Cleveland Fed,forecast update,Forecast publication date,True,inflation feature candidate
cboe_vix,Cboe,market close,"Trading date, after close",True,stress feature candidate


,decision_date,decision_at,next_trade_date,information_lag_days
1179,2026-08-07,2026-08-07 16:00:00-04:00,2026-08-10,3
1180,2026-08-14,2026-08-14 16:00:00-04:00,2026-08-17,3
1181,2026-08-21,2026-08-21 16:00:00-04:00,2026-08-24,3
1182,2026-08-28,2026-08-28 16:00:00-04:00,2026-08-31,3
1183,2026-09-04,2026-09-04 16:00:00-04:00,2026-09-08,4


In [18]:
# 2. Canonical point-in-time macro-event constructors
# ============================================================

FRED_NATIVE_FREQUENCY = {
    "ICSA": "weekly",
    "unrate": "monthly",
    "sahm_realtime": "monthly",
    "cpi_index_sa": "monthly",
    "T10YIE": "daily",
    "DFII10": "daily",
    "infl_5y5y": "daily",
    "fedfunds": "daily",
    "cash_yield_3m": "monthly",
    "NFCI": "weekly",
    "ANFCI": "weekly",
    "HY_OAS": "daily",
    "EPU": "daily",
    "oil_wti": "daily",
    "usd_broad": "weekly",
}

CANONICAL_EVENT_COLUMNS = [
    "source",
    "dataset",
    "series",
    "series_id",
    "observation_date",
    "available_date",
    "valid_to",
    "reference_date",
    "value",
    "native_frequency",
    "record_type",
    "pit_eligible",
    "availability_rule",
    "provenance",
]


def make_fred_events(
    history: pd.DataFrame,
) -> pd.DataFrame:
    events = history.copy()

    events["available_date"] = pd.to_datetime(
        events["realtime_start"],
        errors="raise",
    )
    events["valid_to"] = pd.to_datetime(
        events["realtime_end"],
        errors="raise",
    )

    events["source"] = "FRED/ALFRED"
    events["dataset"] = "fred_alfred"
    events["reference_date"] = pd.NaT
    events["native_frequency"] = events["series"].map(
        FRED_NATIVE_FREQUENCY
    )
    events["record_type"] = "vintage interval"
    events["pit_eligible"] = True
    events["availability_rule"] = "ALFRED realtime_start"
    events["provenance"] = events["series_id"]

    if events["native_frequency"].isna().any():
        missing = events.loc[
            events["native_frequency"].isna(),
            "series",
        ].unique()

        raise ValueError(
            f"Missing FRED frequency metadata: {missing}"
        )

    return events[CANONICAL_EVENT_COLUMNS]


def make_indpro_events(
    history: pd.DataFrame,
) -> pd.DataFrame:
    events = (
        history.rename(
            columns={
                "vintage_date": "available_date",
                "INDPRO": "value",
            }
        )
        .sort_values(
            ["observation_date", "available_date"]
        )
        .reset_index(drop=True)
    )

    previous_value = events.groupby(
        "observation_date"
    )["value"].shift()

    changed = (
        previous_value.isna()
        | events["value"].ne(previous_value)
    )

    events = events.loc[changed].copy()

    next_vintage = events.groupby(
        "observation_date"
    )["available_date"].shift(-1)

    events["valid_to"] = (
        next_vintage - pd.Timedelta(days=1)
    ).fillna(RUN_CUTOFF)

    events["source"] = "Downloaded ALFRED file"
    events["dataset"] = "indpro"
    events["series"] = "INDPRO"
    events["series_id"] = "INDPRO"
    events["reference_date"] = pd.NaT
    events["native_frequency"] = "monthly"
    events["record_type"] = "vintage interval"
    events["pit_eligible"] = True
    events["availability_rule"] = "Vintage-column date"
    events["provenance"] = INDPRO_PATH.name

    return events[CANONICAL_EVENT_COLUMNS]


CLEVELAND_SERIES_NAMES = {
    "cpi_inflation_mom": "cleveland_cpi_nowcast_mom",
    "core_cpi_inflation_mom": "cleveland_core_cpi_nowcast_mom",
    "pce_inflation_mom": "cleveland_pce_nowcast_mom",
    "core_pce_inflation_mom": "cleveland_core_pce_nowcast_mom",
}


def make_cleveland_events(
    nowcasts: pd.DataFrame,
    cpi_calendar: pd.DataFrame,
) -> pd.DataFrame:
    events = (
        nowcasts.reset_index()
        .melt(
            id_vars=[
                "forecast_date",
                "target_month",
                "source_file",
            ],
            value_vars=list(CLEVELAND_SERIES_NAMES),
            var_name="series_id",
            value_name="value",
        )
        .dropna(subset=["value"])
        .rename(
            columns={
                "forecast_date": "available_date",
                "target_month": "observation_date",
            }
        )
    )

    events["series"] = events["series_id"].map(
        CLEVELAND_SERIES_NAMES
    )

    # Map each CPI target month to its first subsequent CPI release.
    release_dates = pd.DatetimeIndex(
        pd.to_datetime(
            cpi_calendar["date"],
            errors="raise",
        )
    ).sort_values().unique()

    cpi_release_map = {}

    for target_month in events["observation_date"].unique():
        target_month = pd.Timestamp(target_month)
        target_end = target_month + pd.offsets.MonthEnd(1)

        later_releases = release_dates[
            release_dates > target_end
        ]

        cpi_release_map[target_month] = (
            later_releases[0]
            if len(later_releases)
            else pd.NaT
        )

    is_cpi_nowcast = events["series_id"].isin(
        [
            "cpi_inflation_mom",
            "core_cpi_inflation_mom",
        ]
    )

    mapped_release_dates = pd.to_datetime(
        events["observation_date"].map(cpi_release_map)
    )

    events["reference_date"] = mapped_release_dates.where(
        is_cpi_nowcast
    )

    # At Friday close, a CPI nowcast is no longer usable once
    # the corresponding CPI release has occurred.
    post_release = (
        is_cpi_nowcast
        & events["reference_date"].notna()
        & events["available_date"].ge(
            events["reference_date"]
        )
    )

    excluded_count = int(post_release.sum())

    events = events.loc[~post_release].copy()

    # Construct update-validity intervals only after excluding
    # post-release observations.
    events = events.sort_values(
        [
            "series",
            "observation_date",
            "available_date",
        ]
    ).reset_index(drop=True)

    next_update = events.groupby(
        ["series", "observation_date"]
    )["available_date"].shift(-1)

    events["valid_to"] = (
        next_update - pd.Timedelta(days=1)
    ).fillna(RUN_CUTOFF)

    is_cpi_nowcast = events["series_id"].isin(
        [
            "cpi_inflation_mom",
            "core_cpi_inflation_mom",
        ]
    )

    has_release_date = (
        is_cpi_nowcast
        & events["reference_date"].notna()
    )

    release_expiry = (
        events["reference_date"]
        - pd.Timedelta(days=1)
    )

    events.loc[has_release_date, "valid_to"] = (
        pd.concat(
            [
                events.loc[has_release_date, "valid_to"],
                release_expiry.loc[has_release_date],
            ],
            axis=1,
        )
        .min(axis=1)
    )

    events["source"] = "Cleveland Fed"
    events["dataset"] = "cleveland_nowcast"
    events["native_frequency"] = "daily forecast update"
    events["record_type"] = "forecast update"
    events["pit_eligible"] = True
    events["availability_rule"] = "Forecast publication date"
    events["provenance"] = events["source_file"]

    if events["valid_to"].lt(
        events["available_date"]
    ).any():
        raise ValueError(
            "Invalid Cleveland nowcast validity intervals remain."
        )

    print(
        f"Excluded {excluded_count:,} post-release "
        "Cleveland CPI nowcast records."
    )

    return events[CANONICAL_EVENT_COLUMNS]


def make_massive_events(
    datasets: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    frames = []

    for endpoint_name, frame in datasets.items():
        long = (
            frame.rename_axis("observation_date")
            .reset_index()
            .melt(
                id_vars="observation_date",
                var_name="series_id",
                value_name="value",
            )
        )

        long["value"] = pd.to_numeric(
            long["value"],
            errors="coerce",
        )

        long = long.dropna(subset=["value"])

        is_daily_pit = endpoint_name in {
            "treasury",
            "funding",
        }

        long["source"] = "Massive"
        long["dataset"] = f"massive_{endpoint_name}"
        long["series"] = (
            "massive_"
            + endpoint_name
            + "_"
            + long["series_id"].astype(str)
        )

        long["available_date"] = (
            long["observation_date"]
            if is_daily_pit
            else pd.NaT
        )

        long["valid_to"] = pd.NaT
        long["reference_date"] = pd.NaT
        long["native_frequency"] = (
            "daily" if is_daily_pit else "monthly"
        )
        long["record_type"] = (
            "observation"
            if is_daily_pit
            else "current-history snapshot"
        )
        long["pit_eligible"] = is_daily_pit
        long["availability_rule"] = (
            "Observation date, end of day"
            if is_daily_pit
            else "Historical release date unavailable"
        )
        long["provenance"] = MASSIVE_ENDPOINTS[
            endpoint_name
        ]

        frames.append(long[CANONICAL_EVENT_COLUMNS])

    return pd.concat(frames, ignore_index=True)

In [19]:
# 3. Build the canonical macro event store
# ============================================================

fred_events = make_fred_events(
    fred_vintage_history
)

indpro_events = make_indpro_events(
    indpro_long
)

cleveland_events = make_cleveland_events(
    inflation_nowcast_long,
    cpi_release_dates,
)

massive_events = make_massive_events(
    massive_raw
)

macro_events = (
    pd.concat(
        [
            fred_events,
            indpro_events,
            cleveland_events,
            massive_events,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "series",
            "observation_date",
            "available_date",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

macro_events["is_missing"] = macro_events[
    "value"
].isna()

model_eligible_events = macro_events.loc[
    macro_events["pit_eligible"]
].copy()

reference_only_events = macro_events.loc[
    ~macro_events["pit_eligible"]
].copy()

Excluded 40 post-release Cleveland CPI nowcast records.


In [20]:
# 4. Canonical market-observation store
# ============================================================

vix_observations = (
    vix_daily.rename_axis("observation_date")
    .reset_index()
    .melt(
        id_vars="observation_date",
        var_name="field",
        value_name="value",
    )
)

vix_observations["source"] = "Cboe"
vix_observations["dataset"] = "cboe_vix"
vix_observations["instrument"] = "VIX"
vix_observations["available_date"] = (
    vix_observations["observation_date"]
)
vix_observations["model_role"] = "stress feature"


tiingo_observations = (
    tiingo_etf_daily.reset_index()
    .melt(
        id_vars=["date", "ticker"],
        var_name="field",
        value_name="value",
    )
    .rename(
        columns={
            "date": "observation_date",
            "ticker": "instrument",
        }
    )
)

tiingo_observations["source"] = "Tiingo"
tiingo_observations["dataset"] = "tiingo_etf"
tiingo_observations["available_date"] = (
    tiingo_observations["observation_date"]
)
tiingo_observations["model_role"] = "evaluation target"


MARKET_COLUMNS = [
    "source",
    "dataset",
    "instrument",
    "field",
    "observation_date",
    "available_date",
    "value",
    "model_role",
]

market_observations = (
    pd.concat(
        [
            vix_observations[MARKET_COLUMNS],
            tiingo_observations[MARKET_COLUMNS],
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "instrument",
            "field",
            "observation_date",
        ]
    )
    .reset_index(drop=True)
)

In [21]:
# 5. Data-layer integrity checks
# ============================================================

assert not macro_events.empty
assert not model_eligible_events.empty
assert not market_observations.empty
assert not decision_calendar.empty

assert model_eligible_events["available_date"].notna().all()

assert model_eligible_events["available_date"].le(
    RUN_CUTOFF
).all()

assert market_observations["available_date"].le(
    RUN_CUTOFF
).all()

assert np.isfinite(
    macro_events["value"].dropna()
).all()

assert np.isfinite(
    market_observations["value"].dropna()
).all()

versioned_events = macro_events.loc[
    macro_events["record_type"].isin(
        ["vintage interval", "forecast update"]
    )
].copy()

assert versioned_events["valid_to"].notna().all()
assert versioned_events["valid_to"].ge(
    versioned_events["available_date"]
).all()

duplicate_event_keys = macro_events.duplicated(
    [
        "source",
        "series",
        "observation_date",
        "available_date",
    ]
)

if duplicate_event_keys.any():
    raise ValueError(
        "Duplicate canonical macro-event keys detected."
    )

versioned_events = versioned_events.sort_values(
    [
        "source",
        "series",
        "observation_date",
        "available_date",
    ]
)

previous_valid_to = versioned_events.groupby(
    [
        "source",
        "series",
        "observation_date",
    ]
)["valid_to"].shift()

overlapping_versions = (
    previous_valid_to.notna()
    & versioned_events["available_date"].le(
        previous_valid_to
    )
)

if overlapping_versions.any():
    raise ValueError(
        "Overlapping point-in-time validity intervals detected."
    )

data_layer_audit = (
    macro_events.groupby(
        [
            "source",
            "dataset",
            "pit_eligible",
        ],
        dropna=False,
    )
    .agg(
        series_count=("series", "nunique"),
        event_count=("series", "size"),
        nonmissing_values=("value", "count"),
        first_observation=("observation_date", "min"),
        last_observation=("observation_date", "max"),
        first_available=("available_date", "min"),
        last_available=("available_date", "max"),
    )
    .sort_index()
)

print(
    f"{len(macro_events):,} canonical macro events | "
    f"{len(model_eligible_events):,} PIT-eligible events | "
    f"{len(decision_calendar):,} weekly decisions"
)

display(data_layer_audit)
display(decision_calendar.tail())

1,033,223 canonical macro events | 1,028,651 PIT-eligible events | 1,184 weekly decisions


series_count  \
source                 dataset              pit_eligible                 
Cleveland Fed          cleveland_nowcast    True                     4   
Downloaded ALFRED file indpro               True                     1   
FRED/ALFRED            fred_alfred          True                    15   
Massive                massive_expectations False                    7   
                       massive_funding      True                    21   
                       massive_inflation    False                    6   
                       massive_labor        False                    4   
                       massive_treasury     True                     7   

                                                          event_count  \
source                 dataset              pit_eligible                
Cleveland Fed          cleveland_nowcast    True                22092   
Downloaded ALFRED file indpro               True                 1644   
FRED/ALFRED            fred_alfred          True               897410   
Massive                massive_expectations False                1904   
                       massive_funding      True                67780   
                       massive_inflation    False                1609   
                       massive_labor        False                1059   
                       massive_treasury     True                39725   

                                                          nonmissing_values  \
source                 dataset              pit_eligible                      
Cleveland Fed          cleveland_nowcast    True                      22092   
Downloaded ALFRED file indpro               True                       1644   
FRED/ALFRED            fred_alfred          True                     896118   
Massive                massive_expectations False                      1904   
                       massive_funding      True                      67780   
                       massive_inflation    False                      1609   
                       massive_labor        False                      1059   
                       massive_treasury     True                      39725   

                                                         first_observation  \
source                 dataset              pit_eligible                     
Cleveland Fed          cleveland_nowcast    True                2013-07-01   
Downloaded ALFRED file indpro               True                2014-01-01   
FRED/ALFRED            fred_alfred          True                2004-01-01   
Massive                massive_expectations False               2004-01-01   
                       massive_funding      True                2004-01-01   
                       massive_inflation    False               2004-01-01   
                       massive_labor        False               2004-01-01   
                       massive_treasury     True                2004-01-02   

                                                         last_observation  \
source                 dataset              pit_eligible                    
Cleveland Fed          cleveland_nowcast    True               2026-09-01   
Downloaded ALFRED file indpro               True               2026-07-01   
FRED/ALFRED            fred_alfred          True               2026-09-10   
Massive                massive_expectations False              2026-08-01   
                       massive_funding      True               2026-09-10   
                       massive_inflation    False              2026-07-01   
                       massive_labor        False              2026-08-01   
                       massive_treasury     True               2026-09-08   

                                                         first_available  \
source                 dataset              pit_eligible                   
Cleveland Fed          cleveland_nowcast    True              2013-08-20   
Downloaded ALFR

,decision_date,decision_at,next_trade_date,information_lag_days
1179,2026-08-07,2026-08-07 16:00:00-04:00,2026-08-10,3
1180,2026-08-14,2026-08-14 16:00:00-04:00,2026-08-17,3
1181,2026-08-21,2026-08-21 16:00:00-04:00,2026-08-24,3
1182,2026-08-28,2026-08-28 16:00:00-04:00,2026-08-31,3
1183,2026-09-04,2026-09-04 16:00:00-04:00,2026-09-08,4


In [22]:
# 6. Reduce version histories to latest-observation frontiers
# ============================================================

def make_asof_candidates(
    events: pd.DataFrame,
) -> pd.DataFrame:
    """
    Reduce canonical events to records that could have been the latest
    observable value for a series at some historical decision date.
    """
    events = events.loc[
        events["pit_eligible"]
    ].copy()

    events["observation_date"] = pd.to_datetime(
        events["observation_date"],
        errors="raise",
    )
    events["available_date"] = pd.to_datetime(
        events["available_date"],
        errors="raise",
    )
    events["valid_to"] = pd.to_datetime(
        events["valid_to"],
        errors="coerce",
    )

    # A current-regime feature cannot represent a future target period.
    # This matters mainly for Cleveland forecasts.
    events["eligible_from"] = events[
        ["observation_date", "available_date"]
    ].max(axis=1)

    versioned = events["record_type"].isin(
        ["vintage interval", "forecast update"]
    )

    valid_interval = (
        ~versioned
        | events["valid_to"].ge(events["eligible_from"])
    )

    events = events.loc[valid_interval].copy()

    events = events.sort_values(
        [
            "series",
            "eligible_from",
            "observation_date",
            "available_date",
        ]
    )

    # Once a newer observation period is available, later revisions to
    # older periods cannot become the current observation again.
    latest_observation_so_far = events.groupby(
        "series"
    )["observation_date"].cummax()

    frontier_mask = (
        ~versioned
        | events["observation_date"].eq(
            latest_observation_so_far
        )
    )

    candidates = events.loc[frontier_mask].copy()

    # If several records become eligible on the same day, retain the
    # newest observation period and then the latest released version.
    candidates = (
        candidates.sort_values(
            [
                "series",
                "eligible_from",
                "observation_date",
                "available_date",
            ]
        )
        .drop_duplicates(
            ["series", "eligible_from"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    return candidates


macro_asof_candidates = make_asof_candidates(
    model_eligible_events
)

candidate_reduction = pd.DataFrame(
    {
        "original_pit_events": (
            model_eligible_events.groupby("series").size()
        ),
        "asof_candidates": (
            macro_asof_candidates.groupby("series").size()
        ),
    }
)

candidate_reduction["retained_pct"] = (
    100
    * candidate_reduction["asof_candidates"]
    / candidate_reduction["original_pit_events"]
)

display(candidate_reduction.sort_values("retained_pct"))

,original_pit_events,asof_candidates,retained_pct
series,,,
ANFCI,377710,795,0.210479
NFCI,338458,795,0.234889
usd_broad,22072,397,1.798659
EPU,117957,3157,2.676399
INDPRO,1644,160,9.732360
oil_wti,6013,798,13.271246
ICSA,5861,895,15.270432
cpi_index_sa,1412,286,20.254958
sahm_realtime,272,84,30.882353


In [23]:
# 7. Generic weekly as-of alignment
# ============================================================

def build_weekly_asof_panels(
    candidates: pd.DataFrame,
    calendar: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
]:
    decisions = (
        calendar[["decision_date"]]
        .drop_duplicates()
        .sort_values("decision_date")
        .reset_index(drop=True)
    )

    decision_index = pd.DatetimeIndex(
        decisions["decision_date"],
        name="decision_date",
    )

    values = pd.DataFrame(index=decision_index)
    observation_dates = pd.DataFrame(index=decision_index)
    available_dates = pd.DataFrame(index=decision_index)

    lineage_parts = []

    for series, right in candidates.groupby(
        "series",
        sort=True,
    ):
        right = (
            right.sort_values("eligible_from")
            .drop_duplicates(
                "eligible_from",
                keep="last",
            )
        )

        aligned = pd.merge_asof(
            decisions,
            right[
                [
                    "eligible_from",
                    "source",
                    "dataset",
                    "series_id",
                    "observation_date",
                    "available_date",
                    "valid_to",
                    "value",
                    "record_type",
                ]
            ],
            left_on="decision_date",
            right_on="eligible_from",
            direction="backward",
            allow_exact_matches=True,
        )

        active = (
            aligned["eligible_from"].notna()
            & (
                aligned["valid_to"].isna()
                | aligned["decision_date"].le(
                    aligned["valid_to"]
                )
            )
        )

        values[series] = (
            aligned["value"].where(active).to_numpy()
        )

        observation_dates[series] = pd.to_datetime(
            aligned["observation_date"].where(active)
        ).to_numpy()

        available_dates[series] = pd.to_datetime(
            aligned["available_date"].where(active)
        ).to_numpy()

        selected = aligned.loc[
            active,
            [
                "decision_date",
                "source",
                "dataset",
                "series_id",
                "observation_date",
                "available_date",
                "valid_to",
                "value",
                "record_type",
            ],
        ].copy()

        selected.insert(1, "series", series)
        lineage_parts.append(selected)

    values = values.sort_index(axis=1)
    observation_dates = observation_dates.reindex(
        columns=values.columns
    )
    available_dates = available_dates.reindex(
        columns=values.columns
    )

    lineage = (
        pd.concat(lineage_parts, ignore_index=True)
        .sort_values(["decision_date", "series"])
        .reset_index(drop=True)
    )

    return (
        values,
        observation_dates,
        available_dates,
        lineage,
    )


(
    weekly_macro_values,
    weekly_macro_observation_dates,
    weekly_macro_available_dates,
    weekly_macro_lineage,
) = build_weekly_asof_panels(
    macro_asof_candidates,
    decision_calendar,
)

In [24]:
# 8. Data-age and release-lag panels
# ============================================================

weekly_macro_observation_age_days = pd.DataFrame(
    index=weekly_macro_values.index,
    columns=weekly_macro_values.columns,
    dtype=float,
)

weekly_macro_publication_age_days = pd.DataFrame(
    index=weekly_macro_values.index,
    columns=weekly_macro_values.columns,
    dtype=float,
)

weekly_macro_release_lag_days = pd.DataFrame(
    index=weekly_macro_values.index,
    columns=weekly_macro_values.columns,
    dtype=float,
)

for series in weekly_macro_values.columns:
    weekly_macro_observation_age_days[series] = (
        weekly_macro_values.index.to_series()
        - weekly_macro_observation_dates[series]
    ).dt.days

    weekly_macro_publication_age_days[series] = (
        weekly_macro_values.index.to_series()
        - weekly_macro_available_dates[series]
    ).dt.days

    weekly_macro_release_lag_days[series] = (
        weekly_macro_available_dates[series]
        - weekly_macro_observation_dates[series]
    ).dt.days

In [25]:
# 9. Weekly market-close panel
# ============================================================

weekly_market_candidates = market_observations.loc[
    (
        market_observations["dataset"].eq("cboe_vix")
        & market_observations["field"].eq("vix_close")
    )
    | (
        market_observations["dataset"].eq("tiingo_etf")
        & market_observations["field"].eq("adj_close")
    )
].copy()

weekly_market_candidates["series"] = np.where(
    weekly_market_candidates["dataset"].eq("cboe_vix"),
    "VIX",
    weekly_market_candidates["instrument"],
)

weekly_market_candidates["series_id"] = (
    weekly_market_candidates["field"]
)
weekly_market_candidates["valid_to"] = pd.NaT
weekly_market_candidates["eligible_from"] = (
    weekly_market_candidates["available_date"]
)
weekly_market_candidates["record_type"] = "market close"

(
    weekly_market_values,
    weekly_market_observation_dates,
    weekly_market_available_dates,
    weekly_market_lineage,
) = build_weekly_asof_panels(
    weekly_market_candidates,
    decision_calendar,
)

In [26]:
# ============================================================
# 10. Weekly as-of panel integrity checks
# ============================================================

assert weekly_macro_values.index.equals(
    pd.DatetimeIndex(
        decision_calendar["decision_date"],
        name="decision_date",
    )
)

assert weekly_market_values.index.equals(
    weekly_macro_values.index
)

assert not weekly_macro_lineage.duplicated(
    ["decision_date", "series"]
).any()

assert not weekly_market_lineage.duplicated(
    ["decision_date", "series"]
).any()

assert weekly_macro_lineage["available_date"].le(
    weekly_macro_lineage["decision_date"]
).all()

assert weekly_macro_lineage["observation_date"].le(
    weekly_macro_lineage["decision_date"]
).all()

assert weekly_market_lineage["available_date"].le(
    weekly_market_lineage["decision_date"]
).all()

assert weekly_macro_observation_age_days.stack().ge(0).all()
assert weekly_macro_publication_age_days.stack().ge(0).all()

weekly_macro_coverage = pd.DataFrame(
    {
        "observations": weekly_macro_values.notna().sum(),
        "coverage_pct": (
            100 * weekly_macro_values.notna().mean()
        ),
        "first_available_week": weekly_macro_values.apply(
            lambda series: series.first_valid_index()
        ),
        "last_available_week": weekly_macro_values.apply(
            lambda series: series.last_valid_index()
        ),
        "median_observation_age_days": (
            weekly_macro_observation_age_days.median()
        ),
        "max_observation_age_days": (
            weekly_macro_observation_age_days.max()
        ),
    }
).sort_values(
    ["coverage_pct", "first_available_week"],
    ascending=[True, True],
)

print(
    f"Weekly macro panel: {weekly_macro_values.shape[0]:,} × "
    f"{weekly_macro_values.shape[1]:,}"
)

print(
    f"Weekly market panel: {weekly_market_values.shape[0]:,} × "
    f"{weekly_market_values.shape[1]:,}"
)

display(weekly_macro_coverage)
display(weekly_macro_values.tail())
display(weekly_market_values.tail())

Weekly macro panel: 1,184 × 48
Weekly market panel: 1,184 × 7


,observations,coverage_pct,first_available_week,last_available_week,median_observation_age_days,max_observation_age_days
HY_OAS,156,13.175676,2023-09-15,2026-09-04,0.0,1.0
massive_funding_interest_on_reserve_balances,267,22.550676,2021-07-30,2026-09-04,0.0,0.0
sahm_realtime,366,30.912162,2019-09-06,2026-09-04,47.0,105.0
usd_broad,396,33.445946,2019-02-08,2026-09-04,7.0,9.0
massive_funding_secured_overnight_financing_rate,440,37.162162,2018-04-06,2026-09-04,0.0,1.0
massive_funding_sofr_25th_percentile,440,37.162162,2018-04-06,2026-09-04,0.0,1.0
massive_funding_sofr_75th_percentile,440,37.162162,2018-04-06,2026-09-04,0.0,1.0
massive_funding_sofr_volume,440,37.162162,2018-04-06,2026-09-04,0.0,1.0
massive_funding_tgcr_25th_percentile,440,37.162162,2018-04-06,2026-09-04,0.0,1.0
massive_funding_tgcr_75th_percentile,440,37.162162,2018-04-06,2026-09-04,0.0,1.0


,ANFCI,DFII10,EPU,HY_OAS,ICSA,INDPRO,NFCI,T10YIE,cash_yield_3m,cleveland_core_cpi_nowcast_mom,...,massive_treasury_yield_1_month,massive_treasury_yield_1_year,massive_treasury_yield_2_year,massive_treasury_yield_30_year,massive_treasury_yield_3_month,massive_treasury_yield_5_year,oil_wti,sahm_realtime,unrate,usd_broad
decision_date,,,,,,,,,,,,,,,,,,,,,
2026-08-07,-0.543,2.43,173.93,2.70,199000.0,102.6395,-0.529,2.25,3.73,0.203008,...,3.79,4.01,4.19,5.19,3.87,4.35,81.96,-0.03,4.1,119.7034
2026-08-14,-0.579,2.39,219.75,2.67,209000.0,102.6395,-0.549,2.27,3.73,0.203342,...,3.79,3.98,4.17,5.25,3.86,4.36,84.77,-0.03,4.1,119.0649
2026-08-21,-0.587,2.35,301.86,2.70,206000.0,102.9939,-0.559,2.34,3.73,0.203342,...,3.80,4.03,4.24,5.27,3.88,4.43,86.48,-0.03,4.1,118.9028
2026-08-28,-0.576,2.34,131.19,2.60,203000.0,102.9939,-0.566,2.31,3.73,0.203342,...,3.84,4.15,4.34,5.22,3.90,4.48,83.90,-0.03,4.1,118.0628
2026-09-04,-0.582,2.42,205.69,2.68,206000.0,102.9939,-0.558,2.35,3.72,0.194477,...,3.79,4.13,4.37,5.24,3.91,4.54,91.48,-0.07,4.1,118.7479


,HYG,IEF,LQD,SPY,TIP,TLT,VIX
decision_date,,,,,,,
2026-08-07,79.174565,92.835305,106.102609,773.26,107.08,82.443111,14.90
2026-08-14,79.274018,92.705772,105.674415,776.34,106.99,81.725868,14.25
2026-08-21,79.174565,92.486562,105.475255,765.72,107.13,81.735830,15.13
2026-08-28,79.303854,92.516455,105.903449,769.35,106.94,82.562652,14.43
2026-09-04,79.160000,92.250000,105.480000,770.19,106.97,82.210000,14.53


In [27]:
# 11. Correct known Massive funding publication lags
# ============================================================

NEXT_BUSINESS_DAY_FUNDING_FIELDS = {
    "effective_fed_funds_rate",
    "effective_fed_funds_volume",
    "overnight_bank_funding_rate",
    "obfr_25th_percentile",
    "obfr_75th_percentile",
    "obfr_volume",
    "secured_overnight_financing_rate",
    "sofr_25th_percentile",
    "sofr_75th_percentile",
    "sofr_volume",
    "tri_party_general_collateral_rate",
    "tri_party_general_collateral_volume",
    "tgcr_25th_percentile",
    "tgcr_75th_percentile",
}

compiled_macro_events = macro_events.copy()

historical_trading_dates = pd.DatetimeIndex(
    etf_adjusted_close["SPY"].dropna().index
).sort_values()


def next_observed_trading_date(
    dates: pd.Series,
    trading_dates: pd.DatetimeIndex,
) -> pd.Series:
    dates = pd.to_datetime(dates, errors="raise")

    positions = trading_dates.searchsorted(
        pd.DatetimeIndex(dates),
        side="right",
    )

    result = np.full(
        len(dates),
        np.datetime64("NaT"),
        dtype="datetime64[ns]",
    )

    valid = positions < len(trading_dates)

    result[valid] = trading_dates[
        positions[valid]
    ].to_numpy()

    return pd.Series(
        result,
        index=dates.index,
        dtype="datetime64[ns]",
    )


next_day_funding = (
    compiled_macro_events["dataset"].eq(
        "massive_funding"
    )
    & compiled_macro_events["series_id"].isin(
        NEXT_BUSINESS_DAY_FUNDING_FIELDS
    )
)

compiled_macro_events.loc[
    next_day_funding,
    "available_date",
] = next_observed_trading_date(
    compiled_macro_events.loc[
        next_day_funding,
        "observation_date",
    ],
    historical_trading_dates,
)

compiled_macro_events.loc[
    next_day_funding,
    "availability_rule",
] = "Next observed trading day"

# Remove tail observations whose publication date is beyond
# the available trading calendar.
unresolved_publication = (
    next_day_funding
    & compiled_macro_events["available_date"].isna()
)

print(
    f"Corrected {next_day_funding.sum():,} funding records | "
    f"removed {unresolved_publication.sum():,} unresolved tail records"
)

compiled_macro_events = compiled_macro_events.loc[
    ~unresolved_publication
].copy()

compiled_model_events = compiled_macro_events.loc[
    compiled_macro_events["pit_eligible"]
].copy()

assert compiled_model_events["available_date"].notna().all()

Corrected 38,364 funding records | removed 0 unresolved tail records


In [28]:
# 12. Rebuild weekly data using corrected availability
# ============================================================

compiled_asof_candidates = make_asof_candidates(
    compiled_model_events
)

(
    weekly_macro_raw,
    weekly_macro_observation_dates_compiled,
    weekly_macro_available_dates_compiled,
    weekly_macro_lineage_compiled,
) = build_weekly_asof_panels(
    compiled_asof_candidates,
    decision_calendar,
)

weekly_macro_publication_age_compiled = pd.DataFrame(
    index=weekly_macro_raw.index,
    columns=weekly_macro_raw.columns,
    dtype=float,
)

weekly_macro_observation_age_compiled = pd.DataFrame(
    index=weekly_macro_raw.index,
    columns=weekly_macro_raw.columns,
    dtype=float,
)

for series in weekly_macro_raw.columns:
    weekly_macro_publication_age_compiled[series] = (
        weekly_macro_raw.index.to_series()
        - weekly_macro_available_dates_compiled[series]
    ).dt.days

    weekly_macro_observation_age_compiled[series] = (
        weekly_macro_raw.index.to_series()
        - weekly_macro_observation_dates_compiled[series]
    ).dt.days

assert weekly_macro_lineage_compiled[
    "available_date"
].le(
    weekly_macro_lineage_compiled["decision_date"]
).all()

In [29]:
# 13. Staleness contracts
# ============================================================

DEFAULT_STALENESS_DAYS = {
    "daily": 7,
    "weekly": 21,
    "monthly": 75,
    "daily forecast update": 7,
}

SERIES_STALENESS_OVERRIDES = {
    "ICSA": 14,
    "INDPRO": 90,
    "NFCI": 21,
    "ANFCI": 21,
    "usd_broad": 21,
    "EPU": 7,
    "oil_wti": 7,
    "HY_OAS": 7,
    "cleveland_cpi_nowcast_mom": 7,
    "cleveland_core_cpi_nowcast_mom": 7,
    "cleveland_pce_nowcast_mom": 7,
    "cleveland_core_pce_nowcast_mom": 7,
}

series_frequency = (
    compiled_asof_candidates[
        ["series", "native_frequency"]
    ]
    .drop_duplicates()
    .set_index("series")["native_frequency"]
)

if series_frequency.index.duplicated().any():
    raise ValueError(
        "Conflicting frequency metadata for a series."
    )


def get_staleness_limit(
    series: str,
) -> int:
    if series in SERIES_STALENESS_OVERRIDES:
        return SERIES_STALENESS_OVERRIDES[series]

    frequency = series_frequency.loc[series]

    if frequency not in DEFAULT_STALENESS_DAYS:
        raise ValueError(
            f"No staleness rule for {series}: {frequency}"
        )

    return DEFAULT_STALENESS_DAYS[frequency]


macro_staleness_limits = pd.Series(
    {
        series: get_staleness_limit(series)
        for series in weekly_macro_raw.columns
    },
    name="maximum_publication_age_days",
)

weekly_macro_stale = (
    weekly_macro_publication_age_compiled.gt(
        macro_staleness_limits,
        axis="columns",
    )
)

weekly_macro_compiled = weekly_macro_raw.mask(
    weekly_macro_stale
)


weekly_market_publication_age = pd.DataFrame(
    index=weekly_market_values.index,
    columns=weekly_market_values.columns,
    dtype=float,
)

for series in weekly_market_values.columns:
    weekly_market_publication_age[series] = (
        weekly_market_values.index.to_series()
        - weekly_market_available_dates[series]
    ).dt.days

MARKET_STALENESS_LIMIT_DAYS = 4

weekly_market_stale = (
    weekly_market_publication_age
    > MARKET_STALENESS_LIMIT_DAYS
)

weekly_market_compiled = weekly_market_values.mask(
    weekly_market_stale
)

In [30]:
# 14. Post-inception data-quality report
# ============================================================

def longest_true_run(
    mask: pd.Series,
) -> int:
    mask = mask.fillna(False).astype(bool)

    if not mask.any():
        return 0

    groups = mask.ne(mask.shift()).cumsum()

    return int(
        mask.groupby(groups).sum().max()
    )


coverage_records = []

for series in weekly_macro_raw.columns:
    first_week = weekly_macro_raw[
        series
    ].first_valid_index()

    if first_week is None:
        continue

    raw = weekly_macro_raw.loc[first_week:, series]
    compiled = weekly_macro_compiled.loc[
        first_week:, series
    ]
    stale = weekly_macro_stale.loc[
        first_week:, series
    ]

    coverage_records.append(
        {
            "series": series,
            "first_week": first_week,
            "last_week": raw.last_valid_index(),
            "eligible_weeks": len(raw),
            "raw_nonmissing_weeks": int(
                raw.notna().sum()
            ),
            "compiled_nonmissing_weeks": int(
                compiled.notna().sum()
            ),
            "post_inception_coverage_pct": (
                100 * compiled.notna().mean()
            ),
            "stale_weeks": int(stale.sum()),
            "longest_missing_run_weeks": (
                longest_true_run(compiled.isna())
            ),
            "median_publication_age_days": (
                weekly_macro_publication_age_compiled
                .loc[first_week:, series]
                .median()
            ),
            "maximum_publication_age_days": (
                macro_staleness_limits[series]
            ),
        }
    )

compiled_coverage = (
    pd.DataFrame(coverage_records)
    .set_index("series")
    .sort_values(
        [
            "post_inception_coverage_pct",
            "first_week",
        ]
    )
)

display(compiled_coverage)

,first_week,last_week,eligible_weeks,raw_nonmissing_weeks,compiled_nonmissing_weeks,post_inception_coverage_pct,stale_weeks,longest_missing_run_weeks,median_publication_age_days,maximum_publication_age_days
series,,,,,,,,,,
massive_funding_fed_overnight_repo_treasury_amount,2004-01-02,2026-09-04,1184,1184,626,52.871622,558,198,2.0,7
massive_funding_fed_overnight_reverse_repo_treasury_amount,2004-01-09,2026-09-04,1183,1183,717,60.608622,466,170,0.0,7
massive_funding_nonfinancial_commercial_paper_90d_rate,2004-01-09,2026-09-04,1183,1183,1027,86.813187,156,11,0.0,7
massive_funding_financial_commercial_paper_90d_rate,2004-01-02,2026-09-04,1184,1184,1167,98.564189,17,5,0.0,7
EPU,2014-03-28,2026-09-04,650,650,643,98.923077,7,4,0.0,7
oil_wti,2011-04-08,2026-09-04,805,805,797,99.006211,8,3,2.0,7
ICSA,2009-05-29,2026-09-04,902,902,896,99.334812,6,6,1.0,14
T10YIE,2014-01-31,2026-09-04,658,657,657,99.848024,0,1,0.0,7
infl_5y5y,2014-01-31,2026-09-04,658,657,657,99.848024,0,1,0.0,7


In [31]:
# 15. Core and extension sample definitions
# ============================================================

CORE_MACRO_POOL = [
    # Growth
    "INDPRO",
    "ICSA",
    "unrate",

    # Inflation
    "cleveland_cpi_nowcast_mom",
    "cleveland_core_cpi_nowcast_mom",
    "cpi_index_sa",
    "T10YIE",
    "infl_5y5y",

    # Financial stress
    "NFCI",
    "ANFCI",

    # Policy and rates
    "fedfunds",
    "DFII10",
    "massive_treasury_yield_3_month",
    "massive_treasury_yield_2_year",
    "massive_treasury_yield_10_year",
]

CORE_AUXILIARY_POOL = [
    "EPU",
    "oil_wti",
]

CORE_REQUIRED_SERIES = [
    "INDPRO",
    "ICSA",
    "cleveland_cpi_nowcast_mom",
    "T10YIE",
    "ANFCI",
    "massive_treasury_yield_2_year",
    "massive_treasury_yield_10_year",
    "VIX",
]

FUNDING_EXTENSION_SERIES = [
    "massive_funding_effective_fed_funds_rate",
    "massive_funding_overnight_bank_funding_rate",
    "massive_funding_secured_overnight_financing_rate",
    "massive_funding_tri_party_general_collateral_rate",
]

REALTIME_2019_EXTENSION_SERIES = [
    "sahm_realtime",
    "usd_broad",
]

RESERVES_EXTENSION_SERIES = [
    "massive_funding_interest_on_reserve_balances",
]

CREDIT_EXTENSION_SERIES = [
    "HY_OAS",
]

ETF_TARGETS = [
    "SPY",
    "IEF",
    "TLT",
    "LQD",
    "HYG",
    "TIP",
]


weekly_input_pool = pd.concat(
    [
        weekly_macro_compiled,
        weekly_market_compiled[["VIX"]],
    ],
    axis=1,
)

weekly_target_prices = weekly_market_compiled[
    ETF_TARGETS
].copy()

if weekly_input_pool.columns.duplicated().any():
    raise ValueError(
        "Duplicate series in weekly input pool."
    )

In [32]:
# 16. Construct sample bundles without deleting calendar weeks
# ============================================================

SAMPLE_SPECS = {
    "core": {
        "additional_series": [],
        "additional_required": [],
    },
    "funding_2018": {
        "additional_series": FUNDING_EXTENSION_SERIES,
        "additional_required": FUNDING_EXTENSION_SERIES,
    },
    "realtime_2019": {
        "additional_series": (
            REALTIME_2019_EXTENSION_SERIES
        ),
        "additional_required": (
            REALTIME_2019_EXTENSION_SERIES
        ),
    },
    "reserves_2021": {
        "additional_series": RESERVES_EXTENSION_SERIES,
        "additional_required": RESERVES_EXTENSION_SERIES,
    },
    "credit_2023": {
        "additional_series": CREDIT_EXTENSION_SERIES,
        "additional_required": CREDIT_EXTENSION_SERIES,
    },
}


def compile_sample(
    sample_name: str,
    specification: dict,
) -> dict:
    macro_columns = list(
        dict.fromkeys(
            CORE_MACRO_POOL
            + CORE_AUXILIARY_POOL
            + specification["additional_series"]
        )
    )

    input_columns = macro_columns + ["VIX"]

    required_columns = list(
        dict.fromkeys(
            CORE_REQUIRED_SERIES
            + specification["additional_required"]
        )
    )

    missing_inputs = set(input_columns).difference(
        weekly_input_pool.columns
    )
    missing_required = set(required_columns).difference(
        weekly_input_pool.columns
    )

    if missing_inputs or missing_required:
        raise KeyError(
            f"{sample_name}: missing inputs "
            f"{sorted(missing_inputs | missing_required)}"
        )

    inputs = weekly_input_pool[input_columns].copy()

    jointly_available = inputs[
        required_columns
    ].notna().all(axis=1)

    if not jointly_available.any():
        raise ValueError(
            f"{sample_name}: no jointly available week."
        )

    sample_start = jointly_available[
        jointly_available
    ].index.min()

    inputs = inputs.loc[sample_start:].copy()
    targets = weekly_target_prices.loc[
        sample_start:
    ].copy()

    valid_week = inputs[
        required_columns
    ].notna().all(axis=1)

    calendar = (
        decision_calendar.set_index("decision_date")
        .reindex(inputs.index)
        .copy()
    )

    return {
        "name": sample_name,
        "start": sample_start,
        "end": inputs.index.max(),
        "inputs": inputs,
        "targets": targets,
        "calendar": calendar,
        "required_series": required_columns,
        "valid_week": valid_week,
    }


compiled_samples = {
    sample_name: compile_sample(
        sample_name,
        specification,
    )
    for sample_name, specification in SAMPLE_SPECS.items()
}

In [33]:
# 17. Compiled-sample audit
# ============================================================

sample_audit_records = []

for sample_name, sample in compiled_samples.items():
    valid_week = sample["valid_week"]

    sample_audit_records.append(
        {
            "sample": sample_name,
            "start": sample["start"],
            "end": sample["end"],
            "calendar_weeks": len(
                sample["inputs"]
            ),
            "input_series": sample[
                "inputs"
            ].shape[1],
            "required_series": len(
                sample["required_series"]
            ),
            "valid_weeks": int(
                valid_week.sum()
            ),
            "valid_week_pct": (
                100 * valid_week.mean()
            ),
            "longest_invalid_run_weeks": (
                longest_true_run(~valid_week)
            ),
            "target_complete_weeks": int(
                sample["targets"].notna().all(
                    axis=1
                ).sum()
            ),
        }
    )

sample_audit = (
    pd.DataFrame(sample_audit_records)
    .set_index("sample")
    .sort_values("start")
)

core_panel = compiled_samples["core"]["inputs"]
core_target_prices = compiled_samples["core"]["targets"]
core_calendar = compiled_samples["core"]["calendar"]
core_valid_week = compiled_samples["core"]["valid_week"]

print(
    f"Core panel: {core_panel.shape[0]:,} weeks × "
    f"{core_panel.shape[1]:,} raw input series"
)

display(sample_audit)
display(core_panel.tail())
display(core_target_prices.tail())

Core panel: 656 weeks × 18 raw input series


,start,end,calendar_weeks,input_series,required_series,valid_weeks,valid_week_pct,longest_invalid_run_weeks,target_complete_weeks
sample,,,,,,,,,
core,2014-02-14,2026-09-04,656,18,8,649,98.932927,6,656
funding_2018,2018-04-06,2026-09-04,440,22,12,434,98.636364,6,440
realtime_2019,2019-09-06,2026-09-04,366,20,10,360,98.360656,6,366
reserves_2021,2021-07-30,2026-09-04,267,19,9,261,97.752809,6,267
credit_2023,2023-09-15,2026-09-04,156,19,9,150,96.153846,6,156


,INDPRO,ICSA,unrate,cleveland_cpi_nowcast_mom,cleveland_core_cpi_nowcast_mom,cpi_index_sa,T10YIE,infl_5y5y,NFCI,ANFCI,fedfunds,DFII10,massive_treasury_yield_3_month,massive_treasury_yield_2_year,massive_treasury_yield_10_year,EPU,oil_wti,VIX
decision_date,,,,,,,,,,,,,,,,,,
2026-08-07,102.6395,199000.0,4.1,0.377373,0.203008,332.568,2.25,2.28,-0.529,-0.543,3.63,2.43,3.87,4.19,4.65,173.93,81.96,14.90
2026-08-14,102.6395,209000.0,4.1,0.345617,0.203342,332.813,2.27,2.30,-0.549,-0.579,3.63,2.39,3.86,4.17,4.68,219.75,84.77,14.25
2026-08-21,102.9939,206000.0,4.1,0.347334,0.203342,332.813,2.34,2.34,-0.559,-0.587,3.63,2.35,3.88,4.24,4.74,301.86,86.48,15.13
2026-08-28,102.9939,203000.0,4.1,0.355918,0.203342,332.813,2.31,2.32,-0.566,-0.576,3.63,2.34,3.90,4.34,4.73,131.19,83.90,14.43
2026-09-04,102.9939,206000.0,4.1,0.382154,0.194477,332.813,2.35,2.33,-0.558,-0.582,3.63,2.42,3.91,4.37,4.78,205.69,91.48,14.53


,SPY,IEF,TLT,LQD,HYG,TIP
decision_date,,,,,,
2026-08-07,773.26,92.835305,82.443111,106.102609,79.174565,107.08
2026-08-14,776.34,92.705772,81.725868,105.674415,79.274018,106.99
2026-08-21,765.72,92.486562,81.735830,105.475255,79.174565,107.13
2026-08-28,769.35,92.516455,82.562652,105.903449,79.303854,106.94
2026-09-04,770.19,92.250000,82.210000,105.480000,79.160000,106.97


In [34]:
# ============================================================
# 18. Export reproducible data-layer snapshot
# ============================================================

import hashlib
import json
import platform
from importlib.metadata import version as package_version


SNAPSHOT_ID = f"as_of_{RUN_CUTOFF:%Y-%m-%d}"

EXPORT_BASE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "macro_regime_v2"
)

EXPORT_DIR = EXPORT_BASE / SNAPSHOT_ID
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def file_sha256(
    file_path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


def export_parquet(
    name: str,
    frame: pd.DataFrame | pd.Series,
) -> dict:
    if isinstance(frame, pd.Series):
        frame = frame.to_frame()

    secret_like_columns = [
        column
        for column in frame.columns.astype(str)
        if any(
            token in column.lower()
            for token in ["api_key", "apikey", "token", "secret"]
        )
    ]

    if secret_like_columns:
        raise ValueError(
            f"{name}: possible credential columns "
            f"{secret_like_columns}"
        )

    output_path = EXPORT_DIR / f"{name}.parquet"
    temporary_path = EXPORT_DIR / f".{name}.parquet.tmp"

    frame.to_parquet(
        temporary_path,
        index=True,
        compression="zstd",
    )

    temporary_path.replace(output_path)

    return {
        "file": output_path.name,
        "rows": int(frame.shape[0]),
        "columns": int(frame.shape[1]),
        "size_bytes": output_path.stat().st_size,
        "sha256": file_sha256(output_path),
    }


EXPORT_OBJECTS = {
    # Canonical event history
    "compiled_macro_events": compiled_macro_events,
    "compiled_asof_candidates": compiled_asof_candidates,
    "weekly_macro_lineage": weekly_macro_lineage_compiled,

    # Weekly macro panels
    "weekly_macro_raw": weekly_macro_raw,
    "weekly_macro_compiled": weekly_macro_compiled,
    "weekly_macro_observation_dates": (
        weekly_macro_observation_dates_compiled
    ),
    "weekly_macro_available_dates": (
        weekly_macro_available_dates_compiled
    ),
    "weekly_macro_observation_age_days": (
        weekly_macro_observation_age_compiled
    ),
    "weekly_macro_publication_age_days": (
        weekly_macro_publication_age_compiled
    ),
    "weekly_macro_stale": weekly_macro_stale,

    # Weekly market and target panels
    "weekly_market_raw": weekly_market_values,
    "weekly_market_compiled": weekly_market_compiled,
    "weekly_market_observation_dates": (
        weekly_market_observation_dates
    ),
    "weekly_market_available_dates": (
        weekly_market_available_dates
    ),
    "weekly_market_publication_age_days": (
        weekly_market_publication_age
    ),
    "weekly_target_prices": weekly_target_prices,

    # Daily market data retained for later return construction
    "tiingo_etf_daily": tiingo_etf_daily,
    "cboe_vix_daily": vix_daily,

    # Contracts and audits
    "decision_calendar": (
        decision_calendar.set_index("decision_date")
    ),
    "dataset_contracts": DATASET_CONTRACTS,
    "series_frequency": series_frequency,
    "macro_staleness_limits": macro_staleness_limits,
    "compiled_coverage": compiled_coverage,
    "sample_audit": sample_audit,
}


manifest = {
    "schema_version": "2.0",
    "snapshot_id": SNAPSHOT_ID,
    "created_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "data_start": DATA_START.date().isoformat(),
    "run_as_of": RUN_CUTOFF.date().isoformat(),
    "last_decision_date": (
        decision_calendar["decision_date"]
        .max()
        .date()
        .isoformat()
    ),
    "decision_frequency": "Weekly, Friday",
    "decision_timing": "Post-close; trade next observed session",
    "python_version": platform.python_version(),
    "pandas_version": package_version("pandas"),
    "numpy_version": package_version("numpy"),
    "artifacts": {},
    "samples": {},
}

for name, frame in EXPORT_OBJECTS.items():
    print(f"Exporting {name}...")
    manifest["artifacts"][name] = export_parquet(
        name,
        frame,
    )


for sample_name, sample in compiled_samples.items():
    sample_objects = {
        f"sample_{sample_name}_inputs": sample["inputs"],
        f"sample_{sample_name}_targets": sample["targets"],
        f"sample_{sample_name}_calendar": sample["calendar"],
        f"sample_{sample_name}_valid_week": (
            sample["valid_week"].rename("valid_week")
        ),
    }

    for artifact_name, frame in sample_objects.items():
        print(f"Exporting {artifact_name}...")
        manifest["artifacts"][artifact_name] = (
            export_parquet(
                artifact_name,
                frame,
            )
        )

    manifest["samples"][sample_name] = {
        "start": sample["start"].date().isoformat(),
        "end": sample["end"].date().isoformat(),
        "calendar_weeks": len(sample["inputs"]),
        "valid_weeks": int(sample["valid_week"].sum()),
        "required_series": sample["required_series"],
    }


manifest_path = EXPORT_DIR / "manifest.json"
temporary_manifest_path = EXPORT_DIR / ".manifest.json.tmp"

temporary_manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

temporary_manifest_path.replace(manifest_path)


latest_pointer = {
    "schema_version": manifest["schema_version"],
    "snapshot_id": SNAPSHOT_ID,
    "manifest": str(
        manifest_path.relative_to(PROJECT_ROOT)
    ),
}

latest_path = EXPORT_BASE / "latest.json"
temporary_latest_path = EXPORT_BASE / ".latest.json.tmp"

temporary_latest_path.write_text(
    json.dumps(
        latest_pointer,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

temporary_latest_path.replace(latest_path)


export_summary = pd.DataFrame(
    manifest["artifacts"]
).T

export_summary["size_mb"] = (
    export_summary["size_bytes"].astype(float)
    / 1024**2
)

print(f"\nSnapshot exported to:\n{EXPORT_DIR}")
print(
    f"Total size: "
    f"{export_summary['size_mb'].sum():,.1f} MB"
)

display(
    export_summary[
        ["rows", "columns", "size_mb"]
    ].sort_values(
        "size_mb",
        ascending=False,
    )
)

Exporting compiled_macro_events...
Exporting compiled_asof_candidates...
Exporting weekly_macro_lineage...
Exporting weekly_macro_raw...
Exporting weekly_macro_compiled...
Exporting weekly_macro_observation_dates...
Exporting weekly_macro_available_dates...
Exporting weekly_macro_observation_age_days...
Exporting weekly_macro_publication_age_days...
Exporting weekly_macro_stale...
Exporting weekly_market_raw...
Exporting weekly_market_compiled...
Exporting weekly_market_observation_dates...
Exporting weekly_market_available_dates...
Exporting weekly_market_publication_age_days...
Exporting weekly_target_prices...
Exporting tiingo_etf_daily...
Exporting cboe_vix_daily...
Exporting decision_calendar...
Exporting dataset_contracts...
Exporting series_frequency...
Exporting macro_staleness_limits...
Exporting compiled_coverage...
Exporting sample_audit...
Exporting sample_core_inputs...
Exporting sample_core_targets...
Exporting sample_core_calendar...
Exporting sample_core_valid_week...
E

,rows,columns,size_mb
compiled_macro_events,1033223,15,6.082823
tiingo_etf_daily,33426,12,1.956518
compiled_asof_candidates,143090,16,1.210388
weekly_macro_lineage,37887,10,0.387726
weekly_macro_available_dates,1184,48,0.299736
weekly_macro_observation_dates,1184,48,0.285551
cboe_vix_daily,9269,4,0.158381
weekly_macro_raw,1184,48,0.131700
weekly_macro_compiled,1184,48,0.131670
weekly_market_observation_dates,1184,7,0.076497
